In [4]:
# awkward_xarray_duck.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Union

import numpy as np
import awkward as ak


Indexer = Union[int, slice, np.ndarray, list]


@dataclass(frozen=True)
class AwkwardDuckArray:
    """
    Minimal xarray duck array wrapper for 1D outer axis selection.

    Contract:
      shape is (len, )
      ndim is 1
      dtype is object
      __getitem__ supports int, slice, 1D int arrays, 1D bool masks
      __array__ is disabled to prevent silent conversion to object arrays
    """
    _ak: ak.Array

    @property
    def shape(self) -> tuple[int]:
        return (len(self._ak),)

    @property
    def ndim(self) -> int:
        return 1

    @property
    def dtype(self) -> np.dtype:
        return np.dtype("O")

    def __len__(self) -> int:
        return len(self._ak)

    # def __getitem__(self, indexer: Indexer) -> Any:
    #     idx = self._normalize_indexer(indexer)
    #     out = self._ak[idx]

    #     # If result is still an Awkward Array, wrap it.
    #     # If it is a scalar/record, return as is.
    #     if isinstance(out, ak.Array):
    #         return AwkwardDuckArray(out)
    #     return out
    def __getitem__(self, indexer):
        # xarray passes tuple keys, even for 1D variables: (indexer,)
        if isinstance(indexer, tuple):
            if len(indexer) != 1:
                raise IndexError(
                    f"Only 1D outer-axis indexing is supported; got tuple of length {len(indexer)}"
                )
            indexer = indexer[0]

        idx = self._normalize_indexer(indexer)
        out = self._ak[idx]

        if isinstance(out, ak.Array):
            return AwkwardDuckArray(out)
        return out

    def __array__(self, dtype: Any = None) -> np.ndarray:
        raise TypeError(
            "AwkwardDuckArray does not allow implicit NumPy conversion. "
            "Use ._ak for Awkward operations, or explicitly materialize if you really need NumPy."
        )

    def __array_function__(self, func, types, args, kwargs):
        return NotImplemented

    def __array_ufunc__(self, ufunc, method, *inputs, **kwargs):
        return NotImplemented

    @staticmethod
    def _normalize_indexer(indexer: Indexer) -> Any:
        if isinstance(indexer, (int, slice)):
            return indexer

        if isinstance(indexer, list):
            indexer = np.asarray(indexer)

        if isinstance(indexer, np.ndarray):
            if indexer.ndim != 1:
                raise IndexError("Only 1D indexers are supported on the outer axis.")
            if indexer.dtype == np.bool_:
                return indexer
            if np.issubdtype(indexer.dtype, np.integer):
                return indexer
            raise IndexError(f"Unsupported indexer dtype: {indexer.dtype}")

        raise IndexError(f"Unsupported indexer type: {type(indexer)!r}")


In [5]:
# test_awkward_xarray_duck.py
import numpy as np
import awkward as ak
import xarray as xr
import pytest

def make_da():
    arr = ak.Array([[1, 2], [], [3]])
    return xr.DataArray(AwkwardDuckArray(arr), dims=("event",))


def test_1_construct():
    da = make_da()
    assert da.dims == ("event",)
    assert da.shape == (3,)


def test_2_isel_scalar():
    da = make_da()
    out = da.isel(event=0).data
    # For scalar selection we return the Awkward element (often a 1D ak.Array like [1,2])
    assert not isinstance(out, AwkwardDuckArray)
    assert ak.to_list(out) == [1, 2]


def test_3_isel_slice():
    da = make_da()
    out = da.isel(event=slice(1, None)).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[], [3]]


def test_4_isel_fancy_int():
    da = make_da()
    out = da.isel(event=[2, 0]).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[3], [1, 2]]


def test_5_isel_bool_mask():
    da = make_da()
    mask = np.array([True, False, True])
    out = da.isel(event=mask).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[1, 2], [3]]


def test_6_numpy_conversion_raises():
    da = make_da()
    with pytest.raises(TypeError):
        np.asarray(da.data)


In [6]:
# test_awkward_xarray_duck.py
import numpy as np
import awkward as ak
import xarray as xr
import pytest


def make_da():
    arr = ak.Array([[1, 2], [], [3]])
    return xr.DataArray(AwkwardDuckArray(arr), dims=("event",))


def test_1_construct():
    da = make_da()
    assert da.dims == ("event",)
    assert da.shape == (3,)


def test_2_isel_scalar():
    da = make_da()
    out = da.isel(event=0).data
    # For scalar selection we return the Awkward element (often a 1D ak.Array like [1,2])
    assert not isinstance(out, AwkwardDuckArray)
    assert ak.to_list(out) == [1, 2]


def test_3_isel_slice():
    da = make_da()
    out = da.isel(event=slice(1, None)).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[], [3]]


def test_4_isel_fancy_int():
    da = make_da()
    out = da.isel(event=[2, 0]).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[3], [1, 2]]


def test_5_isel_bool_mask():
    da = make_da()
    mask = np.array([True, False, True])
    out = da.isel(event=mask).data
    assert isinstance(out, AwkwardDuckArray)
    assert out.shape == (2,)
    assert ak.to_list(out._ak) == [[1, 2], [3]]


def test_6_numpy_conversion_raises():
    da = make_da()
    with pytest.raises(TypeError):
        np.asarray(da.data)


In [7]:
akarr = ak.Array([[1,2], [], [3,4,5]])
duck = AwkwardDuckArray(akarr)

duck.shape   # (3,) regular outer axis
duck[0]      # returns ak element [1,2] (ragged inside is irrelevant here)
duck[[2,0]]  # returns AwkwardDuckArray wrapping [[3,4,5],[1,2]]


AwkwardDuckArray(_ak=<Array [[3, 4, 5], [1, 2]] type='2 * var * int64'>)

In [8]:
# ---- Data ----
event_coord = np.array([101, 102, 103])

ragged = ak.Array([[1, 2], [], [3, 4, 5]])

regular = np.array(
    [[10.0, 11.0],
     [20.0, 21.0],
     [30.0, 31.0]]
)  # shape (event, sensor)

# ---- Dataset with shared regular dimension ----
ds = xr.Dataset(
    data_vars=dict(
        ragged=("event", AwkwardDuckArray(ragged)),
        regular=(("event", "sensor"), regular),
    ),
    coords=dict(
        event=("event", event_coord),
        sensor=("sensor", ["s0", "s1"]),
    ),
)

print(ds)

# # ---- Shared selection by coordinate label ----
# sel_label = ds.sel(event=103)
# print("\nds.sel(event=103)")
# print("ragged:", ak.to_list(sel_label["ragged"].data))
# print("regular:", sel_label["regular"].values)

# ---- Shared selection by positional index ----
sel_pos = ds.isel(event=[0, 2])
print("\nds.isel(event=[0, 2])")
print("ragged:", ak.to_list(sel_pos["ragged"].data._ak))
print("regular:\n", sel_pos["regular"].values)

# ---- Shared selection by boolean mask ----
mask = np.array([True, False, True])
sel_mask = ds.isel(event=mask)
print("\nds.isel(event=mask)")
print("ragged:", ak.to_list(sel_mask["ragged"].data._ak))
print("regular:\n", sel_mask["regular"].values)

<xarray.Dataset> Size: 112B
Dimensions:  (event: 3, sensor: 2)
Coordinates:
  * event    (event) int64 24B 101 102 103
  * sensor   (sensor) <U2 16B 's0' 's1'
Data variables:
    ragged   (event) object 24B AwkwardDuckArray(_ak=<Array [[1, 2], [], [3, ...
    regular  (event, sensor) float64 48B 10.0 11.0 20.0 21.0 30.0 31.0

ds.isel(event=[0, 2])
ragged: [[1, 2], [3, 4, 5]]
regular:
 [[10. 11.]
 [30. 31.]]

ds.isel(event=mask)
ragged: [[1, 2], [3, 4, 5]]
regular:
 [[10. 11.]
 [30. 31.]]


Select values by label instead of integer location: x.loc['2014-01-01'] or x.sel(time='2014-01-01').

In [15]:
ds.ragged.isel(event=[0])

<xarray.DataArray 'ragged' (event: 1)> Size: 8B
AwkwardDuckArray(_ak=<Array [[1, 2]] type='1 * var * int64'>)
Coordinates:
  * event    (event) int64 8B 101

In [1]:
import xarray as xr

ds = xr.open_dataset("report_output/16_03_2026_nb14_prognostics_combined/results.nc")

In [5]:
print(ds)



<xarray.Dataset> Size: 39kB
Dimensions:                (dataset: 1, model: 13, metric_key: 68)
Coordinates:
  * dataset                (dataset) <U4 16B 'NB14'
  * model                  (model) <U82 4kB 'baselines.crossformer_model.Cros...
  * metric_key             (metric_key) <U43 12kB 'test/loss' ... 'val_best_r...
Data variables:
    mean                   (dataset, model, metric_key) float64 7kB ...
    std                    (dataset, model, metric_key) float64 7kB ...
    n                      (dataset, model, metric_key) float64 7kB ...
    sort_metric            (dataset, model) <U25 1kB ...
    opt_metric             (dataset, model) <U8 416B ...
    opt_mode               (dataset, model) float64 104B ...
    opt_value              (dataset, model) float64 104B ...
    total_runs             (dataset, model) float64 104B ...
    configs_failed_seed    (dataset, model) float64 104B ...
    configs_failed_metric  (dataset, model) float64 104B ...
Attributes:
    project_nam